# 03C — Two-component Gaussian-mixture environmental null M3 (Python)

This notebook performs the **Python-side** estimation and audit of the flexible environmental null M3 used in the paper.

M3 replaces the single-Gaussian Probit–Normal environmental null M0 by a two-component Gaussian mixture on the probit scale:

*y_t ~ pi N(mu1, sigma1^2) + (1-pi) N(mu2, sigma2^2),  p_t = Phi(y_t),*

with

*L_t | p_t, n_t ~ Binomial(n_t, p_t).*

The model has five free parameters:

*mu1, sigma1, mu2, sigma2, pi.*

For reporting, the components are ordered so that *mu1 < mu2*.

## Role of this notebook

This notebook is intentionally **Python only**.

It performs:

1. the exact collapsed-boundary audit M3 = M0;
2. Python multi-start search for interior M3 candidates;
3. selection of the best Python candidate relative to M0;
4. export of all Python candidates and diagnostics for later cross-language comparison.

The final manuscript M3 values are **not chosen here**. They are selected in `03D_M3_cross_language_validation.ipynb` after comparing the Python and Julia searches.

This distinction is important because finite-mixture likelihoods can contain nearby local modes. The manuscript therefore refers to the reported M3 estimates as **cross-language validated best-found candidates**, not guaranteed global optima.

## Dependency during the current refactor stage

Before the final module refactor, this notebook uses the current canonical helper module:

`revision_models.py`

After all notebooks are split and verified, AntiGravity can move the shared functions into `modules/` and replace the import without changing the scientific calculations.

## Data policy

The notebook first looks for:

`data/M_1920_2023.csv`

If the proprietary file is unavailable, it falls back to:

`data/M_1920_2023_synthetic.csv`

The synthetic file reproduces the public workflow only; it does not reproduce the manuscript's empirical numerical values.

Outputs are written to `pdata/`.


In [6]:
from pathlib import Path
import math
import random

import numpy as np
import pandas as pd

from IPython.display import display

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
PDATA_DIR = ROOT / "pdata"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PDATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 123
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Current canonical M3 implementation
# ------------------------------------------------------------

try:
    import revision_models as rm
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "\n03C currently expects the canonical `revision_models.py` file "
        "in the project root.\n\n"
        "Copy the current IDMs `revision_models.py` into the working project "
        "before running 03C.\n\n"
        "This dependency will be moved into `modules/` during the final "
        "AntiGravity refactor after the notebook split is complete."
    ) from exc

# ------------------------------------------------------------
# Data selection
# ------------------------------------------------------------

REAL_DATA = DATA_DIR / "M_1920_2023.csv"
SYNTHETIC_DATA = DATA_DIR / "M_1920_2023_synthetic.csv"

if REAL_DATA.exists():
    DATA_FILE = REAL_DATA
    DATA_SOURCE = "REAL"

    print("=" * 72)
    print("DATA SOURCE: proprietary annual Moody's data")
    print(f"File: {DATA_FILE}")
    print("=" * 72)

elif SYNTHETIC_DATA.exists():
    DATA_FILE = SYNTHETIC_DATA
    DATA_SOURCE = "SYNTHETIC"

    print("=" * 72)
    print("DATA SOURCE: FULLY SYNTHETIC annual default data")
    print("The resulting M3 estimates will NOT reproduce the manuscript.")
    print(f"File: {DATA_FILE}")
    print("=" * 72)

else:
    raise FileNotFoundError(
        "\nNo annual default dataset was found.\n\n"
        "If you have authorized access to the proprietary data, place it at:\n"
        "    data/M_1920_2023.csv\n\n"
        "Otherwise first run:\n"
        "    01_generate_synthetic_data.ipynb\n\n"
        "which should create:\n"
        "    data/M_1920_2023_synthetic.csv\n"
    )

df = pd.read_csv(DATA_FILE)

EXPECTED_COLUMNS = [
    "Year",
    "ALL",
    "D_ALL",
    "SG",
    "D_SG",
    "IG",
    "D_IG",
]

missing = [
    c for c in EXPECTED_COLUMNS
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

df = df[EXPECTED_COLUMNS].copy()

for c in EXPECTED_COLUMNS:
    df[c] = pd.to_numeric(
        df[c],
        errors="raise",
    )

df["Year"] = df["Year"].astype(int)

for c in [
    "ALL",
    "D_ALL",
    "SG",
    "D_SG",
    "IG",
    "D_IG",
]:
    df[c] = df[c].astype(int)

print(
    f"Rows: {len(df)} "
    f"| Years: {df['Year'].min()}–{df['Year'].max()}"
)


DATA SOURCE: proprietary annual Moody's data
File: ./data/M_1920_2023.csv
Rows: 104 | Years: 1920–2023


## 1. Read the canonical M0 reference from 03B

03C does not refit M0. It reads the M0 likelihood and parameters produced by `03B_environmental_models.ipynb`.

This keeps the M3 robustness calculation anchored to the exact same M0 reference used in the main environmental-model comparison.


In [7]:
PHASE_03B_COMP = (
    PDATA_DIR
    / "table_iii_environmental_model_comparison.csv"
)

PHASE_03B_PARAMS = (
    PDATA_DIR
    / "environmental_model_parameters.csv"
)

if not PHASE_03B_COMP.exists():
    raise FileNotFoundError(
        f"Missing {PHASE_03B_COMP}.\n"
        "Run 03B_environmental_models.ipynb first."
    )

if not PHASE_03B_PARAMS.exists():
    raise FileNotFoundError(
        f"Missing {PHASE_03B_PARAMS}.\n"
        "Run 03B_environmental_models.ipynb first."
    )

df_03b_comp = pd.read_csv(
    PHASE_03B_COMP
)

df_03b_params = pd.read_csv(
    PHASE_03B_PARAMS
)

required_comp_cols = {
    "class",
    "model",
    "T",
    "k",
    "NLL",
    "AIC",
    "BIC",
    "boundary",
    "solution",
}

missing = (
    required_comp_cols
    - set(df_03b_comp.columns)
)

if missing:
    raise RuntimeError(
        "03B comparison CSV has an unexpected schema. "
        f"Missing columns: {sorted(missing)}"
    )

required_param_cols = {
    "class",
    "model",
    "parameter",
    "estimate",
    "identified",
}

missing = (
    required_param_cols
    - set(df_03b_params.columns)
)

if missing:
    raise RuntimeError(
        "03B parameter CSV has an unexpected schema. "
        f"Missing columns: {sorted(missing)}"
    )

display(
    df_03b_comp[
        df_03b_comp["model"] == "M0"
    ].reset_index(drop=True)
)


,class,model,T,k,NLL,AIC,BIC,boundary,solution
0,ALL,M0,104,2,431.363711,866.727421,872.016203,False,M0
1,SG,M0,104,2,417.708535,839.417070,844.705852,False,M0
2,IG,M0,104,2,182.381686,368.763372,374.052154,False,M0


## 2. Helpers

The annual observations are grouped by identical *(n_t, L_t)* pairs for efficient likelihood evaluation.

The reporting helper enforces the manuscript convention *mu1 < mu2*. If the optimizer returns the labels in the reverse order, the components are swapped and *pi* is replaced by *1-pi*.


In [8]:
def make_grouped_nh(
    series_n,
    series_h,
):
    tmp = pd.DataFrame({
        "n": np.asarray(
            series_n,
            dtype=int,
        ),
        "h": np.asarray(
            series_h,
            dtype=int,
        ),
    })

    grouped = (
        tmp
        .value_counts(
            ["n", "h"]
        )
        .rename("count")
        .reset_index()
        .sort_values(
            ["n", "h"]
        )
        .reset_index(
            drop=True
        )
    )

    return (
        grouped["n"].to_numpy(
            dtype=int
        ),
        grouped["h"].to_numpy(
            dtype=int
        ),
        grouped["count"].to_numpy(
            dtype=int
        ),
    )


def get_m0_row(
    key,
):
    sub = df_03b_comp[
        (
            df_03b_comp[
                "class"
            ] == key
        )
        &
        (
            df_03b_comp[
                "model"
            ] == "M0"
        )
    ]

    if len(sub) != 1:
        raise RuntimeError(
            f"Expected exactly one M0 row for {key}; "
            f"found {len(sub)}."
        )

    return sub.iloc[0]


def get_m0_parameter(
    key,
    parameter,
):
    sub = df_03b_params[
        (
            df_03b_params[
                "class"
            ] == key
        )
        &
        (
            df_03b_params[
                "model"
            ] == "M0"
        )
        &
        (
            df_03b_params[
                "parameter"
            ] == parameter
        )
    ]

    if len(sub) != 1:
        raise RuntimeError(
            f"Expected exactly one M0 parameter "
            f"{parameter} for {key}; found {len(sub)}."
        )

    return float(
        sub.iloc[0][
            "estimate"
        ]
    )


def order_mixture_components(
    mu1,
    sigma1,
    mu2,
    sigma2,
    pi,
):
    """
    Return the same physical mixture with reporting order mu1 < mu2.
    """

    mu1 = float(mu1)
    sigma1 = float(sigma1)
    mu2 = float(mu2)
    sigma2 = float(sigma2)
    pi = float(pi)

    if mu1 <= mu2:
        return {
            "mu1": mu1,
            "sigma1": sigma1,
            "mu2": mu2,
            "sigma2": sigma2,
            "pi": pi,
        }

    return {
        "mu1": mu2,
        "sigma1": sigma2,
        "mu2": mu1,
        "sigma2": sigma1,
        "pi": 1.0 - pi,
    }


def information_criteria(
    nll,
    k,
    T,
):
    nll = float(nll)
    k = int(k)
    T = int(T)

    return {
        "logL": -nll,
        "AIC": 2.0 * nll + 2.0 * k,
        "BIC": 2.0 * nll + k * math.log(T),
    }


## 3. Python M3 settings

The settings below reproduce the Python Phase 2B search logic from the canonical analysis.

The collapsed M3 boundary is checked directly against M0 using 100-point Gauss–Hermite quadrature. Interior candidates are generated by the canonical Python multi-start optimizer in `revision_models.py`.


In [9]:
GH_N_M3 = 100
MAXITER_M3 = 6000

M3_IMPROVEMENT_TOL = 1e-6
COLLAPSED_EQ_TOL = 1e-8

CLASSES = [
    "ALL",
    "SG",
    "IG",
]

print(
    f"GH_N_M3={GH_N_M3}, "
    f"MAXITER_M3={MAXITER_M3}"
)


GH_N_M3=100, MAXITER_M3=6000


## 4. Collapsed-boundary audit and Python multi-start search

The two-component mixture contains M0 in its closure. Therefore, before interpreting any interior M3 candidate, the notebook verifies numerically that

*mu1 = mu2 = mu_M0, sigma1 = sigma2 = sigma_M0*

reproduces the exact M0 likelihood.

The Python interior search is then run independently for ALL, SG, and IG.


In [10]:
comparison_rows = []
parameter_rows = []
diagnostic_rows = []
collapsed_rows = []
best_candidate_rows = []

for key in CLASSES:

    print(
        "\n"
        + "=" * 72
    )
    print(
        f"Python M3 audit: {key}"
    )
    print(
        "=" * 72
    )

    series_n = df[key]
    series_h = df[
        "D_" + key
    ]

    n_arr, h_arr, c_arr = (
        make_grouped_nh(
            series_n,
            series_h,
        )
    )

    T_obs = int(
        np.sum(
            c_arr
        )
    )

    # --------------------------------------------------------
    # M0 reference from 03B
    # --------------------------------------------------------

    m0_row = get_m0_row(
        key
    )

    nll_m0 = float(
        m0_row[
            "NLL"
        ]
    )

    mu0 = get_m0_parameter(
        key,
        "mu",
    )

    sigma0 = get_m0_parameter(
        key,
        "sigma",
    )

    if T_obs != int(
        m0_row[
            "T"
        ]
    ):
        raise RuntimeError(
            f"{key}: current data length differs from 03B."
        )

    # --------------------------------------------------------
    # Exact collapsed M3 = M0 audit
    # --------------------------------------------------------

    nll_collapsed = (
        rm.nll_probit_normal_mixture_physical(
            mu1=mu0,
            mu2=mu0,
            sigma1=sigma0,
            sigma2=sigma0,
            pi=0.5,
            n_arr=n_arr,
            h_arr=h_arr,
            c_arr=c_arr,
            GH_N=GH_N_M3,
        )
    )

    delta_collapsed = abs(
        float(
            nll_collapsed
        )
        - nll_m0
    )

    collapsed_ok = bool(
        delta_collapsed
        < COLLAPSED_EQ_TOL
    )

    collapsed_rows.append({
        "class": key,
        "nll_M0": nll_m0,
        "nll_collapsed_M3":
            float(
                nll_collapsed
            ),
        "abs_delta":
            delta_collapsed,
        "collapsed_equivalence_ok":
            collapsed_ok,
    })

    print(
        f"[{key}] M0 NLL={nll_m0:.10f}"
    )

    print(
        f"[{key}] collapsed M3 NLL="
        f"{float(nll_collapsed):.10f}"
    )

    print(
        f"[{key}] |delta|="
        f"{delta_collapsed:.3e}"
    )

    if not collapsed_ok:
        raise RuntimeError(
            f"{key}: collapsed M3 does not reproduce "
            "the 03B M0 likelihood."
        )

    # --------------------------------------------------------
    # Python multi-start interior search
    # --------------------------------------------------------

    fit_raw = (
        rm.fit_probit_normal_mixture_multistart(
            series_n,
            series_h,
            dataset_class=key,
            GH_N=GH_N_M3,
            maxiter=MAXITER_M3,
        )
    )

    # Preserve every start.
    diagnostic_rows.extend(
        fit_raw[
            "diagnostics"
        ]
    )

    nll_best_interior = float(
        fit_raw[
            "nll_best_interior"
        ]
    )

    interior_improves = bool(
        np.isfinite(
            nll_best_interior
        )
        and (
            nll_best_interior
            < nll_m0
            - M3_IMPROVEMENT_TOL
        )
    )

    # --------------------------------------------------------
    # Recover the best interior candidate
    # --------------------------------------------------------

    if interior_improves:

        if not bool(
            fit_raw[
                "is_collapsed_boundary"
            ]
        ):
            raw_params = {
                "mu1":
                    float(
                        fit_raw[
                            "mu1"
                        ]
                    ),
                "sigma1":
                    float(
                        fit_raw[
                            "sigma1"
                        ]
                    ),
                "mu2":
                    float(
                        fit_raw[
                            "mu2"
                        ]
                    ),
                "sigma2":
                    float(
                        fit_raw[
                            "sigma2"
                        ]
                    ),
                "pi":
                    float(
                        fit_raw[
                            "pi"
                        ]
                    ),
            }

        else:
            diag = pd.DataFrame(
                fit_raw[
                    "diagnostics"
                ]
            )

            nll_col = (
                "nll"
                if "nll" in diag.columns
                else "final_nll"
            )

            diag[nll_col] = (
                pd.to_numeric(
                    diag[nll_col],
                    errors="coerce",
                )
            )

            diag = (
                diag[
                    np.isfinite(
                        diag[
                            nll_col
                        ]
                    )
                ]
                .sort_values(
                    nll_col
                )
                .reset_index(
                    drop=True
                )
            )

            if len(diag) == 0:
                raise RuntimeError(
                    f"{key}: no finite interior diagnostic candidate."
                )

            best = diag.iloc[0]

            raw_params = {
                "mu1":
                    float(
                        best[
                            "final_mu1"
                        ]
                    ),
                "sigma1":
                    float(
                        best[
                            "final_sigma1"
                        ]
                    ),
                "mu2":
                    float(
                        best[
                            "final_mu2"
                        ]
                    ),
                "sigma2":
                    float(
                        best[
                            "final_sigma2"
                        ]
                    ),
                "pi":
                    float(
                        best[
                            "final_pi"
                        ]
                    ),
            }

        ordered = order_mixture_components(
            **raw_params
        )

        nll_m3 = (
            nll_best_interior
        )

        boundary = False
        identified = True

        solution = (
            "interior 2-component mixture"
        )

    else:

        # ----------------------------------------------------
        # Collapsed M0 boundary selected
        # ----------------------------------------------------

        ordered = {
            "mu1": mu0,
            "sigma1": sigma0,
            "mu2": mu0,
            "sigma2": sigma0,
            "pi": 0.5,
        }

        nll_m3 = (
            nll_m0
        )

        boundary = True
        identified = False

        solution = (
            "M0 boundary "
            "(collapsed 2-component mixture)"
        )

    metrics = information_criteria(
        nll=nll_m3,
        k=5,
        T=T_obs,
    )

    comparison_rows.append({
        "class": key,
        "model": "M3",
        "source": "Python multi-start",
        "T": T_obs,
        "k": 5,
        "NLL": float(
            nll_m3
        ),
        "logL": float(
            metrics[
                "logL"
            ]
        ),
        "AIC": float(
            metrics[
                "AIC"
            ]
        ),
        "BIC": float(
            metrics[
                "BIC"
            ]
        ),
        "NLL_M0": nll_m0,
        "delta_NLL_vs_M0":
            float(
                nll_m3
                - nll_m0
            ),
        "boundary": boundary,
        "identified": identified,
        "solution": solution,
    })

    for parameter in [
        "mu1",
        "sigma1",
        "mu2",
        "sigma2",
        "pi",
    ]:
        parameter_rows.append({
            "class": key,
            "model": "M3",
            "source": "Python multi-start",
            "parameter": parameter,
            "estimate":
                float(
                    ordered[
                        parameter
                    ]
                ),
            "identified": identified,
            "boundary": boundary,
            "solution": solution,
        })

    best_candidate_rows.append({
        "class": key,
        "source": "Python",
        "NLL":
            float(
                nll_m3
            ),
        "best_interior_NLL":
            float(
                nll_best_interior
            ),
        "mu1":
            float(
                ordered[
                    "mu1"
                ]
            ),
        "sigma1":
            float(
                ordered[
                    "sigma1"
                ]
            ),
        "mu2":
            float(
                ordered[
                    "mu2"
                ]
            ),
        "sigma2":
            float(
                ordered[
                    "sigma2"
                ]
            ),
        "pi":
            float(
                ordered[
                    "pi"
                ]
            ),
        "boundary": boundary,
        "identified": identified,
        "solution": solution,
    })

    print(
        f"[{key}] best Python interior NLL="
        f"{nll_best_interior:.10f}"
    )

    print(
        f"[{key}] selected Python M3 NLL="
        f"{nll_m3:.10f}"
    )

    print(
        f"[{key}] delta NLL vs M0="
        f"{nll_m3 - nll_m0:.10f}"
    )



Python M3 audit: ALL
[ALL] M0 NLL=431.3637105780
[ALL] collapsed M3 NLL=431.3637105780
[ALL] |delta|=1.501e-11
[ALL] best Python interior NLL=430.2712920906
[ALL] selected Python M3 NLL=430.2712920906
[ALL] delta NLL vs M0=-1.0924184874

Python M3 audit: SG
[SG] M0 NLL=417.7085349447
[SG] collapsed M3 NLL=417.7085349447
[SG] |delta|=2.956e-11
[SG] best Python interior NLL=416.2635553362
[SG] selected Python M3 NLL=416.2635553362
[SG] delta NLL vs M0=-1.4449796085

Python M3 audit: IG
[IG] M0 NLL=182.3816860461
[IG] collapsed M3 NLL=182.3816860461
[IG] |delta|=3.956e-11
[IG] best Python interior NLL=180.1129479180
[IG] selected Python M3 NLL=180.1129479180
[IG] delta NLL vs M0=-2.2687381281


## 5. Python-side result tables

The first table is the Python-only M3 comparison against M0.

The second is the compact candidate table intended as an input to `03D_M3_cross_language_validation.ipynb`.


In [11]:
df_m3_python_comparison = pd.DataFrame(
    comparison_rows
)

df_m3_python_parameters = pd.DataFrame(
    parameter_rows
)

df_m3_python_diagnostics = pd.DataFrame(
    diagnostic_rows
)

df_m3_collapsed_audit = pd.DataFrame(
    collapsed_rows
)

df_m3_python_candidates = pd.DataFrame(
    best_candidate_rows
)

display(
    df_m3_collapsed_audit.round(
        10
    )
)

display(
    df_m3_python_comparison.round(
        8
    )
)

display(
    df_m3_python_candidates.round(
        10
    )
)

assert bool(
    df_m3_collapsed_audit[
        "collapsed_equivalence_ok"
    ].all()
)

for key in CLASSES:

    row = (
        df_m3_python_comparison[
            df_m3_python_comparison[
                "class"
            ] == key
        ]
        .iloc[0]
    )

    assert (
        float(
            row[
                "NLL"
            ]
        )
        <= float(
            row[
                "NLL_M0"
            ]
        )
        + M3_IMPROVEMENT_TOL
    )


,class,nll_M0,nll_collapsed_M3,abs_delta,collapsed_equivalence_ok
0,ALL,431.363711,431.363711,0.0,True
1,SG,417.708535,417.708535,0.0,True
2,IG,182.381686,182.381686,0.0,True


,class,model,source,T,k,NLL,logL,AIC,BIC,NLL_M0,delta_NLL_vs_M0,boundary,identified,solution
0,ALL,M3,Python multi-start,104,5,430.271292,-430.271292,870.542584,883.764539,431.363711,-1.092418,False,True,interior 2-component mixture
1,SG,M3,Python multi-start,104,5,416.263555,-416.263555,842.527111,855.749065,417.708535,-1.444980,False,True,interior 2-component mixture
2,IG,M3,Python multi-start,104,5,180.112948,-180.112948,370.225896,383.447850,182.381686,-2.268738,False,True,interior 2-component mixture


,class,source,NLL,best_interior_NLL,mu1,sigma1,mu2,sigma2,pi,boundary,identified,solution
0,ALL,Python,430.271292,430.271292,-2.580053,0.697085,-2.411760,3.766324e-01,0.199835,False,True,interior 2-component mixture
1,SG,Python,416.263555,416.263555,-3.195875,1.079321,-1.968803,3.424642e-01,0.124965,False,True,interior 2-component mixture
2,IG,Python,180.112948,180.112948,-3.419428,0.499547,-2.608762,5.140000e-08,0.883577,False,True,interior 2-component mixture


## 6. Export Python M3 results for 03D

The compact candidate CSV is the intended interface between 03C and 03D.

Julia should write an analogous candidate table. `03D_M3_cross_language_validation.ipynb` will compare the two implementations and select the cross-language validated best-found candidate for each class.


In [12]:
df_m3_python_diagnostics.to_csv(
    PDATA_DIR
    / "m3_python_diagnostics.csv",
    index=False,
)

df_m3_python_parameters.to_csv(
    PDATA_DIR
    / "m3_python_parameters.csv",
    index=False,
)

df_m3_python_comparison.to_csv(
    PDATA_DIR
    / "m3_python_model_comparison.csv",
    index=False,
)

df_m3_collapsed_audit.to_csv(
    PDATA_DIR
    / "m3_python_collapsed_audit.csv",
    index=False,
)

df_m3_python_candidates.to_csv(
    PDATA_DIR
    / "m3_python_best_candidates.csv",
    index=False,
)

print(
    "Saved Python M3 outputs:"
)

for name in [
    "m3_python_diagnostics.csv",
    "m3_python_parameters.csv",
    "m3_python_model_comparison.csv",
    "m3_python_collapsed_audit.csv",
    "m3_python_best_candidates.csv",
]:
    print(
        "  ",
        PDATA_DIR / name,
    )


Saved Python M3 outputs:
   ./pdata/m3_python_diagnostics.csv
   ./pdata/m3_python_parameters.csv
   ./pdata/m3_python_model_comparison.csv
   ./pdata/m3_python_collapsed_audit.csv
   ./pdata/m3_python_best_candidates.csv


## 7. Manuscript reference values — informational only

The current V3 manuscript reports the **cross-language validated** M3 candidates, after combining the Python and Julia searches:

| Class | NLL | mu1 | sigma1 | mu2 | sigma2 | pi |
|---|---:|---:|---:|---:|---:|---:|
| ALL | 430.271 | -2.580053 | 0.697083 | -2.411763 | 0.376634 | 0.199837 |
| SG | 416.177 | -2.837232 | 0.895836 | -1.962688 | 0.337446 | 0.161096 |
| IG | 180.113 | -3.419428 | 0.499547 | -2.608762 | approximately 5.14e-8 | 0.883577 |

These values are **not asserted as Python-only targets** in this notebook.

In particular, the canonical audit found that the Julia search discovered a lower SG candidate than the earlier Python search. Therefore, a mismatch between the Python-only SG result here and the final manuscript SG value is not by itself a failure.

The final comparison belongs in 03D.


## Output summary

A complete run writes:

### `pdata/`

- `m3_python_diagnostics.csv`
- `m3_python_parameters.csv`
- `m3_python_model_comparison.csv`
- `m3_python_collapsed_audit.csv`
- `m3_python_best_candidates.csv`

No Julia code is executed by this notebook.

This deliberate separation preserves the independence of the cross-language validation.
